In [1]:
import pandas as pd
import numpy as np
import re

In [2]:
data_path = 'event_level_data_dirty.csv'
clean_path = 'event_level_data_clean.csv'
df = pd.read_csv(data_path)
df.head()


,record_id,timestamp,day_of_week,hour_of_day,is_weekend,is_public_holiday,lat,long,weather,temperature,humidity,location_id,location_name,timezone_info
0,1,2025-01-01 07:01:00,2.0,7.0,False,True,1.280884,{}103.869885{},rainy,24.8,93.5,0,SEMBAWANG EATING HOUSE,NaN
1,2,2025-01-01 07:16:00,2.0,7.0,\r\nFalse\r\n,\nTrue\n,1.280884,103.869885,rainy,60.0,91.6,0,SEMBAWANG EATING HOUSE,NaN
2,3,NaN,2.0,7.0,False,NaN,1.280884,103.869885,cloudy,()23.7,86.9,0,SEMBAWANG EATING HOUSE,NaN
3,4,2025-01-01 07:38:00,2.0,7.0,@False,True,1.280884,103.869885,cloudy,24.2,85.7,0,SEMBAWANG EATING HOUSE,NaN
4,5,2025-01-01 07:39:00,NaN,7.0,\r\nFalse\r\n,True,\r\n1.280884\r\n,103.869885,rainy,24.7,91().2,0,SEMBAWANG EATING HOUSE,NaN


In [3]:
def clean_data(input_path, output_path):
    # Load data
    df = pd.read_csv(input_path)

    # Drop duplicates and unnecessary columns
    df = df.drop_duplicates(subset=['record_id'], keep='first')
    if 'timezone_info' in df.columns:
        df = df.drop(columns=['timezone_info'])

    # Clean String Noise
    def basic_clean(text):
        if pd.isna(text): return text
        text = str(text).strip().lower()
        text = re.sub(r'[{}()\[\]@*&|°\\ø$%//#/]', '', text)
        text = text.replace('á', 'a').replace('ë', 'e').replace('ï', 'i').replace('ü', 'u').replace('û', 'ou')
        return text

    for col in ['weather', 'is_weekend', 'is_public_holiday', 'location_name']:
        df[col] = df[col].apply(basic_clean)

    def clean_weather(value):
        if pd.isna(value):
            return 'unknown'
        elif 'night_clear' in value:
            return 'night_clear'
        elif 'rain' in value:
            return 'rainy'
        elif 'cloud' in value:
            return 'cloudy'
        elif 'clear' in value:
            return 'clear'
        else:
            return 'other'
    df['weather'] = df['weather'].apply(clean_weather)

    # Handle Timestamps and Date Features
    # Fill missing timestamps based on sequential record_ids
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    df['timestamp'] = df['timestamp'].interpolate(method='linear')

    # Recalculate time features to ensure consistency
    df['day_of_week'] = df['timestamp'].dt.dayofweek
    df['hour_of_day'] = df['timestamp'].dt.hour
    df['is_weekend'] = df['day_of_week'].isin([5, 6])

    # Clean Numeric Columns (Lat, Long, Temp, Humidity)
    def clean_numeric(val):
        if pd.isna(val): return np.nan
        # Extract digits, dots, and minus signs only
        clean_val = re.sub(r'[^0-9.\-]', '', str(val))
        try:
            return float(clean_val)
        except:
            return np.nan

    num_cols = ['lat', 'long', 'temperature', 'humidity']
    for col in num_cols:
        df[col] = df[col].apply(clean_numeric)

    # Impute Location Data using Location ID
    # Use location_id to fix incorrect Lat/Long/Name (e.g., -100.0 or 999.0)
    for col in ['lat', 'long', 'location_name']:
        # Create a mapping of location_id to the most common (mode) valid value
        mapping = df[df[col].notna() & (df[col] != 0) & (df[col] != -100) & (df[col] != 999)] \
                    .groupby('location_id')[col].agg(lambda x: x.value_counts().index[0])
        df[col] = df['location_id'].map(mapping)

    # Handle Outliers in Environment Data
    # Cap temperatures to reasonable ranges (e.g., 20-40) or interpolate
    df.loc[(df['temperature'] < 15) | (df['temperature'] > 45), 'temperature'] = np.nan
    df['temperature'] = df.groupby('location_id')['temperature'].transform(lambda x: x.interpolate().ffill().bfill())
    df['humidity'] = df.groupby('location_id')['humidity'].transform(lambda x: x.interpolate().ffill().bfill())

    # Final Formatting
    df['is_public_holiday'] = df['is_public_holiday'].map({'true': True, 'false': False}).fillna(False).astype(bool)
    df['location_id'] = df['location_id'].astype(int)

    # Sort and Save
    df = df.sort_values('record_id')
    df.to_csv(output_path, index=False)
    return df

In [4]:
# Run the cleaning
clean_df = clean_data(data_path, clean_path)

/var/folders/27/7dr1zd1n6bj1vvsl3vy6pg1h0000gn/T/ipykernel_34826/1793197447.py:75: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['is_public_holiday'] = df['is_public_holiday'].map({'true': True, 'false': False}).fillna(False).astype(bool)


In [5]:
clean_df.head()

,record_id,timestamp,day_of_week,hour_of_day,is_weekend,is_public_holiday,lat,long,weather,temperature,humidity,location_id,location_name
0,1,2025-01-01 07:01:00,2,7,False,True,1.280884,103.869885,rainy,24.80,93.5,0,sembawang eating house
1,2,2025-01-01 07:16:00,2,7,False,True,1.280884,103.869885,rainy,24.25,91.6,0,sembawang eating house
2,3,2025-01-01 07:27:00,2,7,False,False,1.280884,103.869885,cloudy,23.70,86.9,0,sembawang eating house
3,4,2025-01-01 07:38:00,2,7,False,True,1.280884,103.869885,cloudy,24.20,85.7,0,sembawang eating house
4,5,2025-01-01 07:39:00,2,7,False,True,1.280884,103.869885,rainy,24.70,91.2,0,sembawang eating house


In [6]:
clean_df['is_weekend'].dtype
clean_df['is_public_holiday'].dtype

dtype('bool')

In [7]:
# Changing True/False values to numeric 1/0
clean_df['is_weekend'] = clean_df['is_weekend'].astype(int)
clean_df['is_public_holiday'] = clean_df['is_public_holiday'].astype(int)

In [8]:
clean_df['weather'].unique()

array(['rainy', 'cloudy', 'unknown', 'other', 'clear', 'night_clear'],
      dtype=object)

In [9]:
# Drop unkown(nan) values from weather
clean_df = clean_df[clean_df['weather'] != 'unknown']
clean_df = clean_df[clean_df['weather'] != 'other']
clean_df['weather'].unique()

array(['rainy', 'cloudy', 'clear', 'night_clear'], dtype=object)

In [10]:
# Use dummy variables for weather
weather_map = {'rainy' : 0, 'cloudy': 1, 'night_clear': 2, 'clear': 3}
clean_df['weather_final'] = clean_df['weather'].map(weather_map)
clean_df.head()

,record_id,timestamp,day_of_week,hour_of_day,is_weekend,is_public_holiday,lat,long,weather,temperature,humidity,location_id,location_name,weather_final
0,1,2025-01-01 07:01:00,2,7,0,1,1.280884,103.869885,rainy,24.80,93.5,0,sembawang eating house,0
1,2,2025-01-01 07:16:00,2,7,0,1,1.280884,103.869885,rainy,24.25,91.6,0,sembawang eating house,0
2,3,2025-01-01 07:27:00,2,7,0,0,1.280884,103.869885,cloudy,23.70,86.9,0,sembawang eating house,1
3,4,2025-01-01 07:38:00,2,7,0,1,1.280884,103.869885,cloudy,24.20,85.7,0,sembawang eating house,1
4,5,2025-01-01 07:39:00,2,7,0,1,1.280884,103.869885,rainy,24.70,91.2,0,sembawang eating house,0


In [11]:
# Creating location frequency column for encoding
location_freq = clean_df['location_name'].value_counts()
clean_df['location_freq'] = clean_df['location_name'].map(location_freq)
clean_df.head()

,record_id,timestamp,day_of_week,hour_of_day,is_weekend,is_public_holiday,lat,long,weather,temperature,humidity,location_id,location_name,weather_final,location_freq
0,1,2025-01-01 07:01:00,2,7,0,1,1.280884,103.869885,rainy,24.80,93.5,0,sembawang eating house,0,10213
1,2,2025-01-01 07:16:00,2,7,0,1,1.280884,103.869885,rainy,24.25,91.6,0,sembawang eating house,0,10213
2,3,2025-01-01 07:27:00,2,7,0,0,1.280884,103.869885,cloudy,23.70,86.9,0,sembawang eating house,1,10213
3,4,2025-01-01 07:38:00,2,7,0,1,1.280884,103.869885,cloudy,24.20,85.7,0,sembawang eating house,1,10213
4,5,2025-01-01 07:39:00,2,7,0,1,1.280884,103.869885,rainy,24.70,91.2,0,sembawang eating house,0,10213


In [12]:
# Reverse dictionary: frequency → list of locations
freq_to_locations = (
    clean_df[['location_name', 'location_freq']]
    .drop_duplicates()
    .groupby('location_freq')['location_name']
    .apply(list)
)

freq_to_locations.loc[10213]


['sembawang eating house']

In [13]:
import os

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer

import mlflow
import mlflow.sklearn

from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier


In [14]:
df = clean_df.copy()

print("clean_df rows:", len(df))
print("clean_df columns:", df.columns.tolist())

clean_df rows: 201391
clean_df columns: ['record_id', 'timestamp', 'day_of_week', 'hour_of_day', 'is_weekend', 'is_public_holiday', 'lat', 'long', 'weather', 'temperature', 'humidity', 'location_id', 'location_name', 'weather_final', 'location_freq']


In [15]:
# Create 30 mins time bins
df["time_bin"] = df["timestamp"].dt.floor("30min")

# Quick proof check: should only be 0,30,60,...
print("Minutes in time_bin:", sorted(df["time_bin"].dt.minute.unique())[:12], "...")


Minutes in time_bin: [np.int32(0), np.int32(30)] ...


In [16]:
# Build TARGET y = event_count per (location_id, time_bin)
#    event_count = number of rows in that bin

group_cols = ["location_id", "time_bin"]

df_counts = (
    df.groupby(group_cols)
      .size()
      .reset_index(name="event_count")
)


In [20]:
# Build FEATURES X per (location_id, time_bin)
# aggregate stable features with "first", continuous with "mean"
agg_rules = {}
for col, rule in [
    ("day_of_week", "first"),
    ("hour_of_day", "first"),
    ("is_weekend", "first"),
    ("is_public_holiday", "first"),
    ("temperature", "mean"),
    ("humidity", "mean"),
    ("weather", "first"),  # keep as text -> one-hot later
]:
    if col in df.columns:
        agg_rules[col] = rule

df_feat = (
    df.groupby(group_cols, as_index=False)
      .agg(agg_rules)
)

# Merge features + target
df30 = df_feat.merge(df_counts, on=group_cols, how="inner")
df30 = df30.sort_values(["location_id", "time_bin"]).reset_index(drop=True)

print("Rows (30-min bins):", len(df30))
print(df30.head())

cat_cols = df30.select_dtypes(include=["object", "category"]).columns.tolist()
print("Categorical features:", cat_cols)

Rows (30-min bins): 90377
   location_id            time_bin  day_of_week  hour_of_day  is_weekend  \
0            0 2025-01-01 07:00:00            2            7           0   
1            0 2025-01-01 07:30:00            2            7           0   
2            0 2025-01-01 08:00:00            2            8           0   
3            0 2025-01-01 08:30:00            2            8           0   
4            0 2025-01-01 09:00:00            2            9           0   

   is_public_holiday  temperature   humidity weather  event_count  
0                  1        24.25  90.666667   rainy            3  
1                  1        24.65  90.925000  cloudy            4  
2                  1        25.25  90.750000   rainy            2  
3                  1        24.90  95.200000   rainy            1  
4                  1        24.35  93.616667   rainy            2  
Categorical features: ['weather']


In [21]:
# Create classification labels from event_count (Low/Medium/High)

try:
    df30["crowd_level"] = pd.qcut(
        df30["event_count"],
        q=3,
        labels=["Low", "Medium", "High"]
    )
except ValueError:
    # fallback: cut based on unique ranks
    df30["crowd_level"] = pd.cut(
        df30["event_count"].rank(method="average"),
        bins=3,
        labels=["Low", "Medium", "High"]
    )

y = df30["crowd_level"].astype(str)
label_map = {
    "Low": 0,
    "Medium": 1,
    "High": 2
}

y_encoded = df30["crowd_level"].map(label_map)


In [22]:
# Build X (Keep time/weather/temp/humidity)

num_cols = [c for c in ["day_of_week","hour_of_day","is_weekend","is_public_holiday","temperature","humidity"]
            if c in df30.columns]
cat_cols = ["weather"] if "weather" in df30.columns else []

X = df30[num_cols + cat_cols].copy()

# Convert boolean-ish columns to int
for bcol in ["is_weekend", "is_public_holiday"]:
    if bcol in X.columns:
        X[bcol] = X[bcol].astype(int)